In [181]:
import time
from sage.all import *
import copy

![Schéma du LFSR demandé]("img/exo1_a.png")

# 1 - Attaque générique par compromis temps-mémoire
## Exercice 6 - Création d'un LFSR

In [183]:
L = [[0, 1, 1, 0, 0, 0, 0, 1, 1], [0, 1, 1, 1, 0, 0, 0], [0, 0, 1, 1, 1, 0, 1, 1]]
realStream  = [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0]
k = GF(Integer(5)); type(k)
    
def LFSR_L1(state):
    L1 = state[0]
    new = (L1[0] ^^ L1[5])
    z1 = L1.pop(0)
    L1.append(new)
    state[0]=L1
    return z1,state

def LFSR_L2(state):
    L2 = state[1]
    new = (((L2[6] ^^ L2[4]) ^^ L2[2]) ^^ L2[0])
    z2 = L2.pop(0)
    L2.append(new)
    state[1] = L2
    return z2, state

def LFSR_L3(state):
    L3 = state[2]
    new = (((L3[0] ^^ L3[1]) ^^ L3[6]) ^^ L3[7])
    z3 = L3.pop(0)
    L3.append(new)
    state[2]=L3
    return z3,state
    
def f(z1,z2,z3):
    return (((z1 & z2) ^^ (z2 & z3)) ^^ (z3))

def LFSR_comb(stat,stream):
    z1, stat = LFSR_L1(stat)
    z2, stat = LFSR_L2(stat)
    z3, stat = LFSR_L3(stat)
    z = f(z1,z2,z3)
    new_stream = stream + [z]
    return stat,new_stream

stat = L
stream = []
key = []
for i in range(0,20):
    stat,stream = LFSR_comb(stat,stream)
    print("LFSR : ",stream)
    if realStream[i] != stream[i]:
        print("ERREUR DE FLOT - Vérifiez l'algorithme ")
        print("RLFS : ",realStream)

LFSR :  [0]
LFSR :  [0, 1]
LFSR :  [0, 1, 1]
LFSR :  [0, 1, 1, 0]
LFSR :  [0, 1, 1, 0, 1]
LFSR :  [0, 1, 1, 0, 1, 0]
LFSR :  [0, 1, 1, 0, 1, 0, 1]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1]
LFSR :  [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0]


## Exercice 6 BIS avec GF

In [184]:
L = [[0, 1, 1, 0, 0, 0, 0, 1, 1], [0, 1, 1, 1, 0, 0, 0], [0, 0, 1, 1, 1, 0, 1, 1]]
L = [vector(GF(2), reg).list() for reg in L]     # <- ligne ajoutee : passage en GF(2)

realStream  = [0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0]

def LFSR_L1(state):
    L1 = state[0]
    new = (L1[0] + L1[5])                        # ^ -> +
    z1 = L1.pop(0)
    L1.append(new)
    state[0]=L1
    
    return z1,state

def LFSR_L1_fixed(state):
    new = (state[0] + state[5])                        # ^ -> +
    z1 = state.pop(0)
    state.append(new)
    return z1,state

def LFSR_L2(state):
    L2 = state[1]
    new = L2[6] + L2[4] + L2[2] + L2[0]          # ^ -> +
    z2 = L2.pop(0)
    L2.append(new)
    state[1] = L2
    
    return z2, state

def LFSR_L3(state):
    L3 = state[2]
    new = L3[0] + L3[1] + L3[6] + L3[7]          # ^ -> +
    z3 = L3.pop(0)
    L3.append(new)
    state[2]=L3
    return z3,state
    
def f(z1,z2,z3):
    return (z1 * z2) + (z2 * z3) + (z3)          # & -> * , ^ -> +

def LFSR_comb(stat,stream):
    z1, stat = LFSR_L1(stat)
    z2, stat = LFSR_L2(stat)
    z3, stat = LFSR_L3(stat)
    z = f(z1,z2,z3)
    new_stream = stream + [z]
    return stat,new_stream

stat = copy.deepcopy(L)
stream = []
key = []
for i in range(0,100):
    stat,stream = LFSR_comb(stat,stream)
print(stream)

[0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1]


## Exercice 7 - Attaque exhaustive

In [ ]:
false_state = true
L_TEST = []
while (false_state):
    for L1 in VectorSpace(GF(2),9):
        if not (false_state):
            break;
        for L2 in VectorSpace(GF(2),7):
            if not(false_state):
                break;
            for L3 in VectorSpace(GF(2),8):
                L_TEST = [L1.list(),L2.list(), L3.list()]
                fake_stat = L_TEST
                fake_stream = []
                key = []
                for i in range(0,100):
                    fake_stat,fake_stream = LFSR_comb(fake_stat,fake_stream)
                    if stream[i] != fake_stream[i]:
                        print("ERREUR DE FLOT - Vérifiez l'algorithme ")
                        print("RLFS : ",realStream)
                        false_state = true
                        break;     
                    else:
                        false_state = false
print("L TEST IS : ", L_TEST)


''' Parallelized code 
@parallel
def search_for_L1(L1_vec):
    l1 = L1_vec.list()
    for L2 in VectorSpace(GF(2), 7):
        for L3 in VectorSpace(GF(2), 8):
            L_TEST = [list(l1), L2.list(), L3.list()]
            fake_stat = [list(l1), L2.list(), L3.list()]
            fake_stream = []
            match = True
            for i in range(len(stream)):
                fake_stat, fake_stream = LFSR_comb(fake_stat, fake_stream)
                if stream[i] != fake_stream[-1]:
                    match = False
                    break
            if match:
                return L_TEST
    return None

l1_candidates = list(VectorSpace(GF(2), 9))

t0 = time.time()
i_count = 0
trouve = None
for entry in search_for_L1(l1_candidates):
    (args, kwargs), res = entry
    i_count += 1
    elapsed = time.time() - t0
    print(f"\rItération n°{i_count}/{len(l1_candidates)} — {elapsed:.1f} s écoulées", end="", flush=True)
    if res is not None:
'''

## Exercice 8 - Attaque temps/mémoire

In [186]:
l = 24
N_base = int(sqrt(2 ** l).numerical_approx())
print(N_base)
def gen_state(N) :
    dic = {}
    for i in range (N):
        L1_gen_state = list(VectorSpace(GF(2),9).random_element())
        L2_gen_state = list(VectorSpace(GF(2),7).random_element())
        L3_gen_state = list(VectorSpace(GF(2),8).random_element())
        L_3 = [L1_gen_state,L2_gen_state,L3_gen_state]
        L_3_base = copy.deepcopy([L1_gen_state,L2_gen_state,L3_gen_state])
        stream = []
        state = []
        for j in range(0,50):
            state,stream = LFSR_comb(L_3,stream)
        dic[tuple(stream)] = L_3_base
    return dic

def gen_state_basic(N,init_state):
    stat = copy.deepcopy(init_state)
    stream = []
    key = []
    for i in range(0,N+50):
        stat,stream = LFSR_comb(stat,stream)
    return stream
    
def time_memory_attack_k(N,dic) :
    t0 = time.time()
    i_count = 0
    real_init_state = copy.deepcopy(L)
    stream = gen_state_basic(N,real_init_state)
    fetching_state = true
    for j in range(N-49):
        tuple_stream = tuple(stream[j:j+50])
        if tuple_stream in dic:
            init_stream = tuple_stream
            return {
                "init_state" : dic[init_stream],
                "linked_stream" : init_stream,
                "interval" : j,
            }
        elapsed = time.time() - t0
        #print(f"\rItération n°{i_count}/ — {elapsed:.1f} s écoulées", end="", flush=True)
        i_count += 1
    print("No state found")                 

gen_dic = gen_state(N_base)
print(f"\n {result}")
result_k = time_memory_attack_k(N_base,gen_dic)
print(result_k)

4096

 None
{'init_state': [[1, 0, 0, 1, 0, 1, 1, 1, 0], [1, 0, 1, 0, 1, 0, 0], [0, 0, 1, 1, 0, 0, 0, 0]], 'linked_stream': (1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1), 'interval': 248}


In [187]:
real_init_state = copy.deepcopy(L)
real_stream = gen_state_basic(N_base, real_init_state)
print(real_stream[0:20])

[0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0]


In [188]:
AAAAAAAAA = result_k["init_state"]
print(AAAAAAAAA)
interval = result_k["interval"]
print(interval)
real_stream = gen_state_basic(N_base, real_init_state)
fake_stream = gen_state_basic(N_base-interval, AAAAAAAAA)
truncated_stream = real_stream[interval:]
print(fake_stream[0:20:])
print(truncated_stream[0:20:])
if fake_stream != truncated_stream:
    if real_stream[interval] != truncated_stream[0]:
        print("Logic error")
    else:
        print("Stream are not the same")
else:
    print("Bien joué!!!")
    

[[1, 0, 0, 1, 0, 1, 1, 1, 0], [1, 0, 1, 0, 1, 0, 0], [0, 0, 1, 1, 0, 0, 0, 0]]
248
[1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1]
[1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1]
Bien joué!!!


# 2 - Attaque par correlation

## Exercice 9

In [195]:
real_init_state = copy.deepcopy(L)
newFlow = gen_state_basic(50,real_init_state)
def ex_LFSR_1() :
    stream = []
    all_streams={}
    allStates = VectorSpace(GF(2),9).list()
    for i in allStates:
        for j in range(100):
            z_1, stat = LFSR_L1_fixed(i.list())
            stream.append(z_1)
        all_streams[tuple(i)] = stream
        stream = []
    return all_streams

for k in allstreams:
    
    
        

{(0, 0, 0, 0, 0, 0, 0, 0, 0): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], (1, 0, 0, 0, 0, 0, 0, 0, 0): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], (0, 1, 0, 0, 0, 0, 0, 0, 0): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], (1, 1,